In [ ]:
# Cell 1: Environment Setup and Library Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning Imports (TensorFlow / Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reinforcement Learning & Model Saving
import joblib

# Set plot aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print(f"TensorFlow Version: {tf.__version__}")
print("All required libraries loaded successfully!")

In [ ]:
# Cell 2: Project 1 - Loading and Preprocessing MNIST Dataset
# Load MNIST handwritten digits dataset
mnist = keras.datasets.mnist
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

print(f"Training set shape: {X_train_raw.shape}, Labels shape: {y_train_raw.shape}")
print(f"Test set shape: {X_test_raw.shape}, Labels shape: {y_test_raw.shape}")

# Normalization: Scale pixel values from [0, 255] to [0.0, 1.0]
X_train = X_train_raw.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

# Reshape images for Deep Neural Network input (Flatten 28x28 to 784 vectors)
X_train_flat = X_train.reshape(-1, 28 * 28)
X_test_flat = X_test.reshape(-1, 28 * 28)

# Convert labels to One-Hot Encoding
y_train_cat = keras.utils.to_categorical(y_train_raw, 10)
y_test_cat = keras.utils.to_categorical(y_test_raw, 10)

# Display sample digits
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i in range(5):
    axes[i].imshow(X_train_raw[i], cmap='gray')
    axes[i].set_title(f"Label: {y_train_raw[i]}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Project 1 - Building, Compiling, and Training Deep Neural Network
# Build Deep Neural Network architecture
model_dnn = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(10, activation='softmax')  # 10 output classes (digits 0-9)
])

# Compile the model
model_dnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_dnn.summary()

# Train the model with early stopping
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_dnn = model_dnn.fit(
    X_train_flat, y_train_cat,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Save the trained model
model_dnn.save("mnist_dnn_model.h5")
print("Saved trained model to 'mnist_dnn_model.h5'")

In [ ]:
# Cell 4: Project 1 - Learning Curves and Model Performance Evaluation
test_loss, test_acc = model_dnn.evaluate(X_test_flat, y_test_cat, verbose=0)
print(f"--- MNIST Test Set Accuracy: {test_acc * 100:.2f}% ---")

# Plot Learning Curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax[0].plot(history_dnn.history['loss'], label='Train Loss', color='navy')
ax[0].plot(history_dnn.history['val_loss'], label='Validation Loss', color='crimson')
ax[0].set_title('DNN Training and Validation Loss')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].legend()

# Accuracy plot
ax[1].plot(history_dnn.history['accuracy'], label='Train Accuracy', color='navy')
ax[1].plot(history_dnn.history['val_accuracy'], label='Validation Accuracy', color='crimson')
ax[1].set_title('DNN Training and Validation Accuracy')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].legend()

plt.tight_layout()
plt.show()

# Confusion Matrix
from sklearn.metrics import confusion_matrix
y_pred_probs = model_dnn.predict(X_test_flat, verbose=0)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test_raw, y_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.title('MNIST Handwritten Digit Confusion Matrix')
plt.xlabel('Predicted Digit')
plt.ylabel('Actual Digit')
plt.show()

In [ ]:
# Cell 5: Project 2 - Custom Grid Maze Environment Definition
class MazeEnv:
    """
    Custom 5x5 Grid World Maze Environment:
    0 = Empty Path
    1 = Wall/Obstacle
    S = Start (0, 0)
    G = Goal (4, 4)
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """
    def __init__(self):
        self.grid_size = 5
        self.start_state = (0, 0)
        self.goal_state = (4, 4)
        
        # Define maze layout (1 represents obstacles)
        self.walls = {(1, 1), (1, 3), (2, 1), (3, 3), (3, 1)}
        self.reset()
        
    def reset(self):
        self.agent_pos = self.start_state
        return self._state_to_index(self.agent_pos)
    
    def _state_to_index(self, pos):
        return pos[0] * self.grid_size + pos[1]
    
    def step(self, action):
        r, c = self.agent_pos
        
        # Action mappings
        if action == 0:   # Up
            r = max(0, r - 1)
        elif action == 1: # Down
            r = min(self.grid_size - 1, r + 1)
        elif action == 2: # Left
            c = max(0, c - 1)
        elif action == 3: # Right
            c = min(self.grid_size - 1, c + 1)
            
        new_pos = (r, c)
        
        # Check hitting wall
        if new_pos in self.walls:
            new_pos = self.agent_pos  # Stay in place
            reward = -5
            done = False
        elif new_pos == self.goal_state:
            reward = 100  # High positive reward for goal
            done = True
        else:
            reward = -1   # Step penalty to encourage finding shortest path
            done = False
            
        self.agent_pos = new_pos
        return self._state_to_index(new_pos), reward, done

print("Maze Environment initialized successfully.")

In [ ]:
# Cell 6: Project 2 - Q-Learning Agent Training Loop
env = MazeEnv()

num_states = env.grid_size * env.grid_size  # 25 states
num_actions = 4                             # 4 actions

# Q-Table Initialization
Q_table = np.zeros((num_states, num_actions))

# Hyperparameters
episodes = 500
alpha = 0.1       # Learning Rate
gamma = 0.95      # Discount Factor
epsilon = 1.0     # Exploration Rate
min_epsilon = 0.01
decay_rate = 0.995

rewards_per_episode = []

# Q-Learning Algorithm
for ep in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # Epsilon-Greedy Action Selection
        if np.random.rand() < epsilon:
            action = np.random.choice(num_actions)  # Explore
        else:
            action = np.argmax(Q_table[state])       # Exploit
            
        next_state, reward, done = env.step(action)
        
        # Q-Value Update Rule (Bellman Equation)
        best_next_action = np.argmax(Q_table[next_state])
        td_target = reward + gamma * Q_table[next_state, best_next_action] * (not done)
        Q_table[state, action] += alpha * (td_target - Q_table[state, action])
        
        state = next_state
        total_reward += reward
        
    # Epsilon Decay
    epsilon = max(min_epsilon, epsilon * decay_rate)
    rewards_per_episode.append(total_reward)

# Save Q-Table Model
joblib.dump(Q_table, "q_learning_maze_model.pkl")
print("Q-Learning training complete. Saved Q-Table to 'q_learning_maze_model.pkl'")

In [ ]:
# Cell 7: Project 2 - Rewards Plot and Optimal Path Visualization
# Plot learning curve (Rewards vs Episodes)
plt.figure(figsize=(10, 5))
plt.plot(rewards_per_episode, alpha=0.3, color='gray', label='Raw Episode Reward')
plt.plot(pd.Series(rewards_per_episode).rolling( window=20).mean(), color='crimson', lw=2, label='20-Episode Moving Average')
plt.title('Q-Learning Agent Learning Curve (Reward Optimization)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.legend()
plt.show()

# Extract Optimal Path from Q-Table
state = env.reset()
done = False
path = [(0, 0)]

while not done and len(path) < 25:
    action = np.argmax(Q_table[state])
    next_state, _, done = env.step(action)
    r = next_state // env.grid_size
    c = next_state % env.grid_size
    path.append((r, c))
    state = next_state

print("Optimal Shortest Path Found by Q-Learning Agent:")
print(" -> ".join([str(p) for p in path]))